In [25]:
import sympy as sp
import numpy as np
from sympy.physics.wigner import wigner_3j
from scipy.optimize import root
import matplotlib.pyplot as plt
import math
import copy

def Build_matrices(N_PC):
    """Uses Wigner 3-j symbols to build the system's structure, does not include mu and sigma"""
    N = N_PC + 1

    Variables = [sp.symbols(f'c{i}') for i in range(N)]
    Variables = np.array(Variables)

    # Construct matrices using Wigner 3j symbols
    matrices = []
    for i in range(N):
        A = [[0 for _ in range(N)] for _ in range(N)]
        for j in range(N):
            for k in range(N):
                symbol = sp.physics.wigner.wigner_3j(i, j, k, 0, 0, 0)
                A[j][k] = sp.Rational(2*(symbol**2))
        A = np.matrix(A)
        matrices.append(A)

    # Build polynomial system F = V^T M V for each matrix
    F = [(Variables @ matrices[i] @ Variables.T).item() for i in range(N)]

    return matrices, F, Variables


def Build_system(H, mu, s, Variables):
    """Creates the quadratic system associated to the saddle node bifurcation with Uniform PC expansion, where N = N_PC+1,
    mu is the expected value and s is sqrt(3)*sigma, with sigma^2 the variance."""
    N = len(Variables)-1
    G = copy.deepcopy(H)

    # Add constraints
    
    G[0] -= 2*mu*Variables[0]+sp.Rational(2,3)*s*Variables[1]      # Constraint for c0
    G[0] = G[0].expand()
    for i in range(1,N):
        G[i] -= sp.Rational(2,2*i+1)*mu*Variables[i]+sp.Rational(2*s*(i+1),(2*i+1)*(2*i+3))*Variables[i+1]+sp.Rational(2*s*i,(2*i-1)*(2*i+1))*Variables[i-1]
        G[i] = G[i].expand()
    G[N] -= sp.Rational(2,2*N+1)*mu*Variables[N]+sp.Rational(2*s*N,(2*N-1)*(2*N+1))*Variables[N-1] # Constraint for c1
    G[N] = G[N].expand()
    return G

def Triangular_Groebner(F, variables, order='grlex'):
    """
    Calcola una base di Gröbner e la trasforma in forma triangolare (tipo "a scalini").
    
    Args:
        F: Lista di polinomi
        variables: Lista ordinata delle variabili
        order: Ordinamento monomiale iniziale (default: 'grlex')
    
    Returns:
        Lista di polinomi in forma triangolare e monica
    """
    G = sp.groebner(F, *variables, order=order)
    triangular = list(G.fglm('lex'))[::-1]  # Reverse: variabili "alte" per prime

    # Rende ciascun polinomio monico
    monic_triangular = []
    for poly in triangular:
        p = sp.Poly(poly, *variables)
        monic = p / p.LC()
        monic_triangular.append(monic.as_expr())
    return monic_triangular


def solve_groebner_triangular_system(groebner_basis, variables, real_only=True):
    """
    Risolve un sistema triangolare di equazioni polinomiali usando sostituzione ricorsiva.
    
    Args:
        groebner_basis: Lista di polinomi in forma triangolare (da Triangular_Groebner)
        variables: Lista ordinata delle variabili
        real_only: Se True, filtra solo soluzioni reali
        
    Returns:
        Lista di dizionari {variabile: valore}
    """
    if not groebner_basis:
        return [{}]  # Nessuna equazione => soluzione vuota (valida)

    current_poly = groebner_basis[0]
    vars_in_poly = current_poly.free_symbols

    if not vars_in_poly:
        # Polinomio costante: se non nullo, il sistema è inconsistente
        return [] if current_poly != 0 else solve_groebner_triangular_system(groebner_basis[1:], variables)

    # Trova la variabile da risolvere (ultima tra le variabili presenti)
    current_var = [v for v in variables if v in vars_in_poly][-1]

    # Trova tutte le radici (simboliche se possibile)
    poly = sp.Poly(current_poly, current_var)
    roots = poly.nroots(maxsteps=500)
    print(current_var, roots)

    # Filtra soluzioni reali se richiesto
    solutions = []
    for r in roots:
        r_val = r.evalf() if hasattr(r, 'evalf') else r
        if not real_only or sp.im(r_val) == 0:
            solutions.append(sp.re(r_val))

    if not solutions:
        return []

    remaining_basis = groebner_basis[1:]
    all_solutions = []

    for sol_val in solutions:
        substituted_basis = [poly.subs(current_var, sol_val) for poly in remaining_basis]

        # Verifica se c'è una contraddizione immediata (es. 1 == 0)
        inconsistent = any(p.is_number and not sp.simplify(p).is_zero for p in substituted_basis)
        if inconsistent:
            continue

        partial_solutions = solve_groebner_triangular_system(substituted_basis, variables, real_only)
        for psol in partial_solutions:
            sol_dict = {current_var: sol_val}
            sol_dict.update(psol)
            all_solutions.append(sol_dict)

    return all_solutions

In [26]:
N_PC = 2

mu = sp.Rational(1, 1)
s = sp.Rational(1, 5)

solutions = []

# Build the necessary matrices and system
matrices, F, Variables = Build_matrices(N_PC)

G = Build_system(F, mu, s, Variables)
print(G)
H = Triangular_Groebner(G, Variables[::-1])
print(H)
solutions = solve_groebner_triangular_system(H, Variables)
print(solutions)
print(len(solutions))


[2*c0**2 - 2*c0 + 2*c1**2/3 - 2*c1/15 + 2*c2**2/5, 4*c0*c1/3 - 2*c0/15 + 8*c1*c2/15 - 2*c1/3 - 4*c2/75, 4*c0*c2/5 + 4*c1**2/15 - 4*c1/75 + 4*c2**2/35 - 2*c2/5]
[c0**8 - 4*c0**7 + 1187*c0**6/180 - 347*c0**5/60 + 35237377*c0**4/12150000 - 5031127*c0**3/6075000 + 489923743*c0**2/3936600000 - 5976719*c0/787320000, 9541983789000*c0**7/642308333 - 33396943261500*c0**6/642308333 + 429825216150*c0**5/5892737 - 33635013247125*c0**4/642308333 + 324700019508798*c0**3/16057708325 - 63636488853822*c0**2/16057708325 + 4908861785234*c0/16057708325 + c1, -3713197950000*c0**7/642308333 + 12996192825000*c0**6/642308333 - 164771313750*c0**5/5892737 + 12409700934375*c0**4/642308333 - 8940244567077*c0**3/1284616666 + 3174315613731*c0**2/2569233332 - 224316922077*c0/2569233332 + c2]
c0 [0, 0.208865101992724, 0.285395325036392, 0.342707723642854, 0.657292276357146, 0.714604674963608, 0.791134898007276, 1.00000000000000]
c1 [0]
c2 [0]
c1 [-0.488490404445741]
c2 [0.515429363673519]
c1 [0.601270130609294]
c2 [0